In [5]:
import os
import pandas as pd
import pdfplumber

In [12]:
def save_table_to_csv(df, output_dir, file_name):
    """Saves tables in CSV fomat in the file system."""
    os.makedirs(output_dir, exist_ok=True)
    full_path = os.path.join(output_dir, file_name)

    if os.path.exists(full_path):
        print(f"Skipping: {file_name} already exists in {output_dir}")
        return False # Signal that we didn't save anything

    df.to_csv(full_path, index=False)
    print(f"Successfully saved: {full_path}")
    return True

In [13]:
def extract_tables_from_pdf(file_path, output_dir, target_pages):
    """Main orchestrator for opening the PDF and processing pages."""
    with pdfplumber.open(file_path) as pdf:
        for page_index, file_name in target_pages.items():
            page = pdf.pages[page_index]
            table = page.extract_table()
            
            if not table:
                print(f"Warning: No table found on page {page_index + 1}")
                continue

            # Convert to DataFrame (skipping header row for data)
            df = pd.DataFrame(table[1:], columns=table[0])

            # Clean column names and data
            df.columns = [str(c).replace('\n', ' ') if c else f"Unnamed_{i}" 
                          for i, c in enumerate(df.columns)]
            df = df.replace('\n', ' ', regex=True)
            
            # Delegate the saving logic to the new function
            save_table_to_csv(df, output_dir, file_name)

In [14]:
file_path = "data/original_data/participation_data/2024-sfia-topline.pdf"
output_dir = os.path.join("data", "extracted_data")
target_pages = {
    31: "SFIA_Aerobic_Activities.csv",
    32: "SFIA_Conditioning_Activities.csv",
    33: "SFIA_Team_Sports.csv"
}

extract_tables_from_pdf(file_path, output_dir, target_pages)

Skipping: SFIA_Aerobic_Activities.csv already exists in data\extracted_data
Skipping: SFIA_Conditioning_Activities.csv already exists in data\extracted_data
Skipping: SFIA_Team_Sports.csv already exists in data\extracted_data


In [8]:
def filter_by_code(main_table, mapping_table, target_cols_list, code_col_name="Code"):
    valid_codes = mapping_table[code_col_name].values

    mask = main_table[target_cols_list].isin(valid_codes).any(axis=1)

    return main_table[mask]

In [15]:
def bulk_filter_by_code(main_tables_list, mapping_table, target_cols_list, code_col_name="Code"):
    filtered_tables = []

    for table in main_tables_list:
        filtered_table = filter_by_code(table, mapping_table, target_cols_list, code_col_name)
        filtered_tables.append(filtered_table)
        
    return filtered_tables

In [9]:
main_table = pd.read_excel('data/original_data/injury_data/NEISS_2024.XLSX')
mapping_table = pd.read_csv('data/original_data/codes/NEISS_tables/product_codes.csv')

filtered_table = filter_by_code(main_table, mapping_table, ['Product_1', 'Product_2', 'Product_3'])

print(main_table.shape)
print(filtered_table.shape)

(361672, 25)
(98267, 25)


In [16]:
def replace_code_with_meaning(main_table, mapping_table, target_cols, meaning_col, code_col="Code"):
    # Ensure new object will be returned
    new_table = main_table.copy()
    mapping_dict = dict(zip(mapping_table[code_col], mapping_table[meaning_col]))

    # Apply the mapping to all target columns
    for col in target_cols:
        new_table[col] = new_table[col].map(mapping_dict).fillna(new_table[col])
    
    return new_table

In [17]:
def bulk_replace_codes_with_meanings(main_tables_list, mapping_table, target_cols, meaning_col, code_col="Code"):
    replaced_data_tables = []

    for table in main_tables_list:
        replaced_table = replace_code_with_meaning(table, mapping_table, target_cols, meaning_col, code_col)
        replaced_data_tables.append(replaced_table)

    return replaced_data_tables

## Clean product codes
The product codes are from the sport section of the NEISS site, but it still have some rows that are still not sport related. A good example for theis are codes in the swimming related section. So let's check this table and remove the unnecssery data

In [ ]:
sport_codes = pd.read_csv("data/original_data/codes/NEISS_tables/product_codes.csv")
column_data = sport_codes[sport_codes["Category"] == "SWIMMING ACTIVITY, POOLS, EQUIPMENT"]["Sport"]

print(column_data)

80    AboveE-Ground Swomming Pools (excl Portable Po...
81                              BuilT-In Swimming Pools
82                              Diving Or Diving Boards
83    Floatinf Toys (excl Official LifE-Saving Devices)
84                              Portable Swimming Pools
85        Scuba Diving (activity, Apparel Or Equipment)
86            Swimming (activity, Apparel Or Equipment)
87                              Swimming Pool Equipment
88                                 Swimming Pool Slides
89                        Swimming Pools, Not Specified
90                                         Wading Pools
91          Water Polo (activity, Apparel Or Equipment)
Name: Sport, dtype: object


In [ ]:
edited_sport_codes = sport_codes.copy()
rows_to_remove = [80, 81, 83, 84, 88, 89, 90]

edited_sport_codes = edited_sport_codes.drop(rows_to_remove, errors='ignore')


In [21]:
sport_codes = pd.read_csv("data/original_data/codes/NEISS_tables/product_codes.csv")
column_data = sport_codes[sport_codes["Category"] == "NONPOWDER GUNS, BBS & PELLETS"]["Sport"]

print(column_data)

60                                    Bb's Or Pellets
61                   Gas, Air Or SprinG-Operated Guns
62    Skeet Shooting (activity, Apparel Or Equipment)
Name: Sport, dtype: object


Here we'll live only the Skeet Shooting, since the others have a chance to don't be related to sport in all of the cases

In [ ]:
rows_to_remove = [60, 61]

edited_sport_codes = edited_sport_codes.drop(rows_to_remove, errors='ignore')

In [ ]:
sport_codes = pd.read_csv("data/original_data/codes/NEISS_tables/product_codes.csv")
column_data = sport_codes[sport_codes["Category"] == "EXERCISE & EQUIPMENT"]["Sport"]

print(column_data)

In [ ]:
edited_sport_codes = edited_sport_codes.drop(29, errors='ignore')

In [22]:
sport_codes = pd.read_csv("data/original_data/codes/NEISS_tables/product_codes.csv")
column_data = sport_codes[sport_codes["Category"] == "ATV'S, MOPEDS, MINIBIKES, ETC."]["Sport"]

print(column_data)

1      All Terr. Veh. (more Than 4 Wheels; Exclusive...
2     All Terrain Vehicles (# Of Wheels Unspecified/...
3      All Terrain Vehicles (four Wheels/off Road Only)
4      All Terrain Vehicles (four Wheels/ofF Road Only)
5                            Dune Buggies/beach Buggies
6                Electric PoweR-Assisted Pedal Bicycles
7                                              GO-Carts
8                                    Minibikes, Powered
9                       Mopeds Or PoweR-Assisted Cycles
10           Motorized Vehicles, Nec (3 Or More Wheels)
11    PedaL-Powered Adult Vehicles (three Or More Wh...
12      PoweR-Assisted Cycles, Not Elsewhere Classified
13              TwO-Wheeled, Powered, OfF-Road Vehicles
14                                     Utility Vehicles
Name: Sport, dtype: object


In [ ]:
edited_sport_codes.loc[4, "Sport"] = edited_sport_codes.loc[3, "Sport"]

In [23]:
column_data = sport_codes[sport_codes["Category"] == "SKATEBOARDS, SCOOTERS, HOVERBOARDS"]["Sport"]

print(column_data)

67      Hoverboards And Powered Skateboards
68          Scooters / Skateboards, Powered
69                        Scooters, Powered
70                     Scooters Unspecified
71    Skateboards, Unpowered Or Unspecified
Name: Sport, dtype: object


In [ ]:
edited_sport_codes.loc[69, "Sport"] = edited_sport_codes.loc[68, "Sport"]